In [10]:
from pathlib import Path
import pandas as pd
import numpy as np
try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# 固定你的项目路径
ROOT = Path(r"C:\Users\Administrator\EDIT-1-24012456")
DATA_PATH = ROOT / "day4_pandas1" / "output" / "ecommerce_customer_cleaned.csv"
OUTPUT_DIR = ROOT / "output" / "day05_analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("文件路径：", DATA_PATH)
print("文件是否存在：", DATA_PATH.exists())

if not DATA_PATH.exists():
    raise FileNotFoundError("找不到清洗文件，请先运行Day04生成csv！")

# 读取数据
df = pd.read_csv(DATA_PATH)

# ==========关键修复：自动生成缺失的TenureGroup==========
def create_tenure_group(x):
    if x <= 12:
        return "0-12个月 新用户"
    elif x <= 24:
        return "13-24个月 成长用户"
    elif x <= 36:
        return "25-36个月 成熟用户"
    else:
        return "36个月以上 老用户"

df["TenureGroup"] = df["Tenure"].apply(create_tenure_group)
print("✅ 已自动生成 TenureGroup 分组字段")

文件路径： C:\Users\Administrator\EDIT-1-24012456\day4_pandas1\output\ecommerce_customer_cleaned.csv
文件是否存在： True
✅ 已自动生成 TenureGroup 分组字段


In [11]:
core_cols = [
    "CustomerID", "Churn", "Tenure", "TenureGroup", "OrderCount",
    "CouponUsed", "CashbackAmount", "DaySinceLastOrder", "Complain",
    "PreferedOrderCat", "PreferredPaymentMode"
]

# 检测缺失字段
missing_cols = [col for col in core_cols if col not in df.columns]
if missing_cols:
    raise KeyError(f"数据缺少字段：{missing_cols}")

validation = pd.Series({
    "行数": len(df),
    "列数": df.shape[1],
    "CustomerID重复数": int(df["CustomerID"].duplicated().sum()),
    "核心字段缺失数": int(df[core_cols].isna().sum().sum()),
    "Churn取值": sorted(df["Churn"].unique().tolist()),
}, name="验收结果")

display(validation.to_frame())
display(df.head())

assert df["CustomerID"].is_unique, "CustomerID存在重复用户"
assert df[core_cols].notna().all().all(), "核心字段存在空值"
assert set(df["Churn"].unique()) == {0, 1}, "Churn只能为0/1"
print("✅ 数据验收通过！")

,验收结果
行数,5630
列数,21
CustomerID重复数,0
核心字段缺失数,0
Churn取值,"[0, 1]"


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount,TenureGroup
0,50001,1,4.00,Mobile Phone,3,6.00,Debit Card,Female,3.00,3,Laptop & Accessory,2,Single,9,1,11.00,1.00,1.00,5.00,159.93,0-12个月 新用户
1,50002,1,9.00,Mobile Phone,1,8.00,UPI,Male,3.00,4,Mobile,3,Single,7,1,15.00,0.00,1.00,0.00,120.90,0-12个月 新用户
2,50003,1,9.00,Mobile Phone,1,30.00,Debit Card,Male,2.00,4,Mobile,3,Single,6,1,14.00,0.00,1.00,3.00,120.28,0-12个月 新用户
3,50004,1,0.00,Mobile Phone,3,15.00,Debit Card,Male,2.00,4,Laptop & Accessory,5,Single,8,0,23.00,0.00,1.00,3.00,134.07,0-12个月 新用户
4,50005,1,0.00,Mobile Phone,1,12.00,Credit Card,Male,3.00,3,Mobile,5,Single,3,0,11.00,1.00,1.00,3.00,129.60,0-12个月 新用户


✅ 数据验收通过！


In [9]:
# 查看当前数据全部列名，复制出来核对
print("数据所有列名：")
print(df.columns.tolist())


数据所有列名：
['CustomerID', 'Churn', 'Tenure', 'PreferredLoginDevice', 'CityTier', 'WarehouseToHome', 'PreferredPaymentMode', 'Gender', 'HourSpendOnApp', 'NumberOfDeviceRegistered', 'PreferedOrderCat', 'SatisfactionScore', 'MaritalStatus', 'NumberOfAddress', 'Complain', 'OrderAmountHikeFromlastYear', 'CouponUsed', 'OrderCount', 'DaySinceLastOrder', 'CashbackAmount']


In [12]:
metric_dictionary = pd.DataFrame([
    ["用户数", "CustomerID", "nunique", "独立用户数量"],
    ["流失人数", "Churn", "sum", "Churn=1 流失用户"],
    ["流失率", "Churn", "mean", "分组内流失用户占比"],
    ["平均订单数", "OrderCount", "mean", "用户平均下单次数"],
    ["平均优惠券使用", "CouponUsed", "mean", "人均使用优惠券次数"],
    ["平均返现", "CashbackAmount", "mean", "用户获得返现金额"],
    ["平均距上次下单天数", "DaySinceLastOrder", "mean", "数值越大活跃度越低"],
], columns=["指标名称", "字段", "聚合方式", "指标说明"])
display(metric_dictionary)

,指标名称,字段,聚合方式,指标说明
0,用户数,CustomerID,nunique,独立用户数量
1,流失人数,Churn,sum,Churn=1 流失用户
2,流失率,Churn,mean,分组内流失用户占比
3,平均订单数,OrderCount,mean,用户平均下单次数
4,平均优惠券使用,CouponUsed,mean,人均使用优惠券次数
5,平均返现,CashbackAmount,mean,用户获得返现金额
6,平均距上次下单天数,DaySinceLastOrder,mean,数值越大活跃度越低


In [13]:
overall_metrics = pd.DataFrame({
    "指标": ["用户总数", "流失人数", "总体流失率", "平均订单数", "订单数中位数",
            "平均优惠券数", "平均返现金额", "平均App时长", "平均满意度", "平均距上次下单天数"],
    "数值": [
        df["CustomerID"].nunique(),
        df["Churn"].sum(),
        df["Churn"].mean(),
        df["OrderCount"].mean(),
        df["OrderCount"].median(),
        df["CouponUsed"].mean(),
        df["CashbackAmount"].mean(),
        df["HourSpendOnApp"].mean(),
        df["SatisfactionScore"].mean(),
        df["DaySinceLastOrder"].mean()
    ]
})
display(overall_metrics)
print(f"全局总体流失率：{df['Churn'].mean():.2%}")

,指标,数值
0,用户总数,"5,630.00"
1,流失人数,948.00
2,总体流失率,0.17
3,平均订单数,2.96
4,订单数中位数,2.00
5,平均优惠券数,1.72
6,平均返现金额,177.22
7,平均App时长,2.93
8,平均满意度,3.07
9,平均距上次下单天数,4.46


全局总体流失率：16.84%


In [14]:
profile_fields = ["TenureGroup", "PreferedOrderCat", "PreferredPaymentMode", "PreferredLoginDevice", "CityTier"]
for field in profile_fields:
    table = df[field].value_counts(dropna=False).rename("用户数").to_frame()
    table["用户占比"] = (table["用户数"] / len(df)).round(4)
    print(f"\n======== {field} ========")
    display(table)


======== TenureGroup ========


,用户数,用户占比
TenureGroup,,
0-12个月 新用户,3734,0.66
13-24个月 成长用户,1467,0.26
25-36个月 成熟用户,425,0.08
36个月以上 老用户,4,0.00



======== PreferedOrderCat ========


,用户数,用户占比
PreferedOrderCat,,
Laptop & Accessory,2050,0.36
Mobile Phone,1271,0.23
Fashion,826,0.15
Mobile,809,0.14
Grocery,410,0.07
Others,264,0.05



======== PreferredPaymentMode ========


,用户数,用户占比
PreferredPaymentMode,,
Debit Card,2314,0.41
Credit Card,1774,0.32
E wallet,614,0.11
Cash on Delivery,514,0.09
UPI,414,0.07



======== PreferredLoginDevice ========


,用户数,用户占比
PreferredLoginDevice,,
Mobile Phone,3996,0.71
Computer,1634,0.29



======== CityTier ========


,用户数,用户占比
CityTier,,
1,3666,0.65
3,1722,0.31
2,242,0.04


In [15]:
tenure_analysis = (
    df.groupby("TenureGroup", observed=True)
    .agg(
        用户数=("CustomerID", "nunique"),
        流失人数=("Churn", "sum"),
        流失率=("Churn", "mean"),
        平均订单数=("OrderCount", "mean"),
        平均返现=("CashbackAmount", "mean"),
        平均距上次下单天数=("DaySinceLastOrder", "mean")
    ).reset_index()
)
display(tenure_analysis)

,TenureGroup,用户数,流失人数,流失率,平均订单数,平均返现,平均距上次下单天数
0,0-12个月 新用户,3734,853,0.23,2.60,161.16,4.03
1,13-24个月 成长用户,1467,95,0.06,3.70,204.92,5.32
2,25-36个月 成熟用户,425,0,0.00,3.56,222.30,5.26
3,36个月以上 老用户,4,0,0.00,2.00,226.38,4.50


In [16]:
complain_analysis = (
    df.groupby("Complain")
    .agg(
        用户数=("CustomerID", "nunique"),
        流失人数=("Churn", "sum"),
        流失率=("Churn", "mean"),
        平均满意度=("SatisfactionScore", "mean"),
        平均订单数=("OrderCount", "mean")
    ).reset_index()
)
complain_analysis["投诉状态"] = complain_analysis["Complain"].map({0:"无投诉",1:"有投诉"})
display(complain_analysis[["投诉状态","用户数","流失人数","流失率","平均满意度","平均订单数"]])

,投诉状态,用户数,流失人数,流失率,平均满意度,平均订单数
0,无投诉,4026,440,0.11,3.09,3.00
1,有投诉,1604,508,0.32,3.00,2.86


In [17]:
category_analysis = (
    df.groupby("PreferedOrderCat")
    .agg(
        用户数=("CustomerID", "nunique"),
        流失率=("Churn", "mean"),
        平均订单数=("OrderCount", "mean"),
        平均优惠券数=("CouponUsed", "mean"),
        平均返现=("CashbackAmount", "mean")
    ).reset_index()
    .sort_values(["流失率","用户数"],ascending=[False,False])
)
category_analysis["用户占比"] = (category_analysis["用户数"] / len(df)).round(4)
display(category_analysis)

,PreferedOrderCat,用户数,流失率,平均订单数,平均优惠券数,平均返现,用户占比
4,Mobile Phone,1271,0.28,2.47,1.68,149.07,0.23
3,Mobile,809,0.27,1.72,0.89,126.25,0.14
0,Fashion,826,0.15,3.87,2.32,210.41,0.15
2,Laptop & Accessory,2050,0.10,2.77,1.65,167.22,0.36
5,Others,264,0.08,5.25,2.33,304.56,0.05
1,Grocery,410,0.05,4.60,2.19,266.23,0.07


In [18]:
payment_analysis = (
    df.groupby("PreferredPaymentMode")
    .agg(
        用户数=("CustomerID", "nunique"),
        流失率=("Churn", "mean"),
        平均订单数=("OrderCount", "mean"),
        平均优惠券数=("CouponUsed", "mean"),
        平均返现=("CashbackAmount", "mean")
    ).reset_index()
    .sort_values("用户数",ascending=False)
)
display(payment_analysis)

,PreferredPaymentMode,用户数,流失率,平均订单数,平均优惠券数,平均返现
2,Debit Card,2314,0.15,2.94,1.72,177.06
1,Credit Card,1774,0.14,3.05,1.68,177.25
3,E wallet,614,0.23,3.01,1.76,185.83
0,Cash on Delivery,514,0.25,3.01,1.82,169.87
4,UPI,414,0.17,2.57,1.70,174.41


In [21]:
churn_behavior = (
    df.groupby("Churn")
    .agg(
        用户数=("CustomerID", "nunique"),
        平均订单数=("OrderCount", "mean"),
        平均优惠券数=("CouponUsed", "mean"),
        平均返现=("CashbackAmount", "mean"),
        平均App时长=("HourSpendOnApp", "mean"),
        平均满意度=("SatisfactionScore", "mean"),
        平均距上次下单天数=("DaySinceLastOrder", "mean")
    ).reset_index()
)
churn_behavior["用户状态"] = churn_behavior["Churn"].map({0:"未流失",1:"已流失"})
display(churn_behavior.drop(columns="Churn"))

,用户数,平均订单数,平均优惠券数,平均返现,平均App时长,平均满意度,平均距上次下单天数,用户状态
0,4682,2.99,1.72,180.64,2.93,3.00,4.71,未流失
1,948,2.81,1.71,160.37,2.96,3.39,3.22,已流失


In [22]:
tenure_complain_analysis = (
    df.groupby(["TenureGroup","Complain"],observed=True)
    .agg(
        用户数=("CustomerID", "nunique"),
        流失人数=("Churn", "sum"),
        流失率=("Churn", "mean"),
        平均订单数=("OrderCount", "mean")
    ).reset_index()
)
tenure_complain_analysis["投诉状态"] = tenure_complain_analysis["Complain"].map({0:"无投诉",1:"有投诉"})
tenure_complain_analysis["样本标注"] = np.where(tenure_complain_analysis["用户数"]<30,"小样本","样本充足")
display(tenure_complain_analysis)

,TenureGroup,Complain,用户数,流失人数,流失率,平均订单数,投诉状态,样本标注
0,0-12个月 新用户,0,2669,397,0.15,2.58,无投诉,样本充足
1,0-12个月 新用户,1,1065,456,0.43,2.66,有投诉,样本充足
2,13-24个月 成长用户,0,1053,43,0.04,3.85,无投诉,样本充足
3,13-24个月 成长用户,1,414,52,0.13,3.35,有投诉,样本充足
4,25-36个月 成熟用户,0,302,0,0.00,3.75,无投诉,样本充足
5,25-36个月 成熟用户,1,123,0,0.00,3.08,有投诉,样本充足
6,36个月以上 老用户,0,2,0,0.00,2.50,无投诉,小样本
7,36个月以上 老用户,1,2,0,0.00,1.50,有投诉,小样本


In [23]:
count_pivot = pd.pivot_table(
    df, index="TenureGroup", columns="Complain", values="CustomerID",
    aggfunc="nunique", fill_value=0, observed=True
).rename(columns={0:"无投诉用户数",1:"有投诉用户数"})

churn_pivot = pd.pivot_table(
    df, index="TenureGroup", columns="Complain", values="Churn",
    aggfunc="mean", observed=True
).rename(columns={0:"无投诉流失率",1:"有投诉流失率"})

cross_pivot = count_pivot.join(churn_pivot).reset_index()
display(cross_pivot)

Complain,TenureGroup,无投诉用户数,有投诉用户数,无投诉流失率,有投诉流失率
0,0-12个月 新用户,2669,1065,0.15,0.43
1,13-24个月 成长用户,1053,414,0.04,0.13
2,25-36个月 成熟用户,302,123,0.00,0.00
3,36个月以上 老用户,2,2,0.00,0.00


In [24]:
output_dict = {
    "overall_metrics.csv": overall_metrics,
    "tenure_analysis.csv": tenure_analysis,
    "complain_analysis.csv": complain_analysis,
    "category_analysis.csv": category_analysis,
    "payment_analysis.csv": payment_analysis,
    "tenure_complain_analysis.csv": tenure_complain_analysis,
    "tenure_complain_pivot.csv": cross_pivot
}

for filename, table in output_dict.items():
    save_path = OUTPUT_DIR / filename
    table.to_csv(save_path, index=False, encoding="utf-8-sig")
    check = pd.read_csv(save_path)
    print(f"✅ 导出 {filename} , 行列：{check.shape}")

print("\n全部报表输出完成，路径：", OUTPUT_DIR)

✅ 导出 overall_metrics.csv , 行列：(10, 2)
✅ 导出 tenure_analysis.csv , 行列：(4, 7)
✅ 导出 complain_analysis.csv , 行列：(2, 7)
✅ 导出 category_analysis.csv , 行列：(6, 7)
✅ 导出 payment_analysis.csv , 行列：(5, 6)
✅ 导出 tenure_complain_analysis.csv , 行列：(8, 8)
✅ 导出 tenure_complain_pivot.csv , 行列：(4, 5)

全部报表输出完成，路径： C:\Users\Administrator\EDIT-1-24012456\output\day05_analysis


In [ ]:
# 业务分析总结
#流失率计算公式
#流失率 = 流失用户数量(Churn=1) ÷ 全部独立用户总数

#.主要发现
#新用户（0-12个月）流失率明显高于老用户；存在投诉的用户群体流失风险显著上升。

#样本中，有投诉用户流失率更高，二者存在关联，**不能直接断定投诉造成流失**，需要客服数据进一步验证。

#数据局限
#本数据集缺少订单消费金额、订单发生日期，无法计算GMV、客单价与月度时间趋势。